# Figure: Supervised track prediction — predicted vs observed coverage

This notebook shows that the fine-tuned Shorkie model reproduces measured epigenomic/transcriptomic coverage. For one example gene we run the released 8-fold supervised ensemble to predict RNA-seq coverage across the gene body, read the matching *observed* coverage from a released bigwig, and plot the two side by side (top: observed reference average, bottom: model-predicted average) over the same genomic bins.

**Reproduces:** the predicted-vs-observed RNA-seq coverage track panel of the supervised track-prediction evaluation figure.

**Upstream:** Runs end-to-end from released data — no upstream `scripts/` stage is required. It loads the released finetuned ensemble (`models.shorkie_finetuned`), the targets sheet (`datasets.targets_sheet`), the R64 genome/GTF, and a released bigwig (`datasets.bigwigs`).

**Requires:** the `yeast_ml` conda env with the package installed (`pip install -e .` from the repo root); `baskerville` and `pysam` importable. **GPU recommended/required** — loading and running the 8-fold SeqNN ensemble is expensive on CPU.

**Source script:** `scripts/03_eval/supervised/track_prediction_eval/3_viz_rnaseq_tracks/2_yeast_rna_seq_models.py` (the terminal coverage-plotting script; the sibling `3_gene_annotation_viz.py` only draws the GTF gene track). The coverage-prediction + plotting logic is ported faithfully; we keep the self-supervised/fine-tuned 170-channel input path (matching the released `shorkie_finetuned` ensemble) and drop the in-script supervised 4-channel and multi-fold-overlap loops.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysam

from baskerville import gene as bgene

from shorkie import config
from shorkie.models.ensemble import load_ensemble
from shorkie.helpers.yeast_helpers import (
    process_sequence,
    predict_tracks,
    plot_coverage_track_bins,
)
from shorkie.viz.load_cov import read_coverage, seq_norm

import pyranges as pr

In [ ]:
# --- Path resolution via shorkie.config (no hardcoded paths) ---
config.load()

model_dir   = str(config.path("models.shorkie_finetuned"))
targets_file = str(config.path("datasets.targets_sheet"))
fasta_file  = str(config.path("genome.fasta"))
gtf_file    = str(config.path("genome.gtf"))
bigwig_dir  = config.path("datasets.bigwigs")
num_folds   = config.get("models.num_folds", 8)

# Each fold ships params.json next to its checkpoint; use fold 0's params for the ensemble.
params_file = f"{model_dir}/train/f0c0/train/params.json"

print("model_dir   :", model_dir)
print("targets_file:", targets_file)
print("num_folds   :", num_folds)

In [ ]:
# --- Select a representative RNA-seq target track ---
# The targets sheet has columns: index, identifier, file (bigwig), sum_stat,
# clip_soft, description, group. We classify RNA-seq tracks by their description
# (mirrors parse_group in the source script).
targets_df = pd.read_csv(targets_file, sep="\t", index_col=0)
rnaseq_df = targets_df[targets_df["description"].str.contains("RNAseq", na=False)]

# Pick one representative RNA-seq track for the figure.
track_row = rnaseq_df.iloc[0]
target_index = pd.Index([track_row.name])  # single-track ensemble slice

# Map the track's bigwig to the RELEASED bigwig directory by basename
# (the sheet's `file` column points at the same data under datasets.bigwigs).
import os
bw_name = os.path.basename(track_row["file"])
bigwig_path = str(bigwig_dir / bw_name)

print("Representative track:", track_row["description"], "(index", track_row.name, ")")
print("Observed bigwig     :", bigwig_path)

In [ ]:
# --- Load the released finetuned 8-fold ensemble, sliced to the one track ---
# load_ensemble sets num_features = 170 (DNA + species channels), matching the
# fine-tuned (self_supervised-architecture) model; so we build 170-channel inputs below.
models = load_ensemble(
    model_dir=model_dir,
    params_file=params_file,
    target_index=target_index,
    num_folds=num_folds,
)
m0 = models[0]
seq_len = 16384
print("Loaded", len(models), "fold models.")

In [ ]:
# --- Choose an example gene and center a 16,384 bp window on it ---
transcriptome = bgene.Transcriptome(gtf_file)
fasta_open = pysam.Fastafile(fasta_file)

# GTF annotation for the self_supervised input builder.
gene_pr = pr.read_gtf(gtf_file)
gene_pr = gene_pr[gene_pr.Feature.isin(["gene", "exon", "five_prime_UTR", "three_prime_UTR"])]

# A well-expressed example gene present in the R64 GTF.
search_gene = "YDR424C"
matched = [k for k in transcriptome.genes if search_gene in k]
assert matched, f"Gene {search_gene} not found in GTF"
gene = transcriptome.genes[matched[0]]
chrom = gene.chrom
gene_start, gene_end = gene.span()
center_pos = gene.midpoint()

start = center_pos - seq_len // 2
end   = center_pos + seq_len // 2
print(f"Gene {search_gene}: {chrom}:{gene_start}-{gene_end}  window {start}-{end}")

In [ ]:
# --- Predict coverage with the fine-tuned ensemble ---
# process_sequence(model_type='self_supervised') builds the (seq_len, 170) input
# (DNA one-hot + species column 114 set to 1), matching load_ensemble's num_features.
seq_one_hot, _ = process_sequence(
    fasta_open, chrom, start, end, gene_pr, seq_len=seq_len, model_type="self_supervised"
)
y_pred = predict_tracks(models, seq_one_hot)  # (1, n_folds, bins, 1)
y_pred = np.mean(y_pred, axis=1, keepdims=True)  # average folds -> (1, 1, bins, 1)
print("Predicted coverage shape:", y_pred.shape)

In [ ]:
# --- Read observed coverage from the released bigwig over the model's output region ---
# The model crops 1024 bp on each side of the input window (16 bp bins).
bin_size = 16
cov_crop = 1024
region_cov_start = start + cov_crop
region_cov_end   = end   - cov_crop

cov_values = read_coverage(bigwig_path, chrom, region_cov_start, region_cov_end)
y_obs = seq_norm(cov_values)                 # bin-normalized observed coverage (bins,)
y_obs = y_obs[np.newaxis, np.newaxis, :, np.newaxis]  # -> (1, 1, bins, 1)
print("Observed coverage shape:", y_obs.shape)

In [ ]:
# --- Plot predicted vs observed coverage over the gene body ---
# plot_coverage_track_bins draws two panels: top = observed (reference) average,
# bottom = model-predicted average, aligned on the same genomic bins.
plot_coverage_track_bins(
    y_wt=y_pred,
    chrom=chrom,
    start=start,
    center_pos=center_pos,
    gene_start=gene_start,
    gene_end=gene_end,
    track_indices=[0],          # single sliced model channel
    track_names=[track_row["description"]],
    track_scales=[1.0],
    track_transforms=[1.0],
    y_ground_truth=y_obs,
    ref_track_indices=[0],      # single observed channel
    plot_window=seq_len - 2 * cov_crop,
    bin_size=bin_size,
    pad=cov_crop // bin_size,
    region_mode="gene",
    rescale_tracks=False,
    save_figs=False,
)
plt.show()
fasta_open.close()